# 04. Multimodal fusion: concat + MLP (Phase 4)

Nối vector PhoBERT `<s>` (768) và vector pooled CLIP (768) thành 1536 chiều, qua MLP `1536 → 512 → 4`.
So sánh với baseline một modality **cùng chế độ train**, để chênh lệch đến từ việc kết hợp modality chứ không phải từ việc đóng băng hay fine-tune encoder:

| Config | Modality | PhoBERT | CLIP | Vai trò |
|---|---|---|---|---|
| `text_only_frozen` | text | đóng băng | — | mốc so sánh |
| `image_only_frozen` | image | — | đóng băng | mốc so sánh |
| `fusion_concat_frozen` | concat | đóng băng | đóng băng | Phase 4: chỉ train MLP |
| `fusion_concat_ftlast` | concat | mở 2 block cuối | mở 2 block cuối + `post_layernorm` | Phase 4: fine-tune nhẹ |

Ba config frozen dùng `lr_head = 1e-3`; `fusion_concat_ftlast` thêm `lr_encoder = 2e-5` cho các block được mở. Còn lại giống baseline Phase 3: CE, không class weight, sampler random, batch 16, tối đa 10 epoch, early stopping patience 3.

Kaggle: bật **Internet** và **GPU T4**. Add Input:
1. `hhhoang/vimmsd-dataset`: ảnh và nhãn gốc
2. `funalee/vimmsd-data-preprocessed`: `text_preprocessed.json` (caption đã tách từ, `scripts/train.py` tự copy sang cache)
3. (tuỳ chọn) output của notebook 02 / 03 có thư mục `text_only/`, `image_only/`: bảng so sánh tự thêm baseline fine-tune toàn bộ của Phase 3

Output trong `/kaggle/working/outputs/<tên config>/`: `best.pt`, `test_metrics.json`, `best_val_metrics.json`, `confusion_matrix_test.png`, `test_predictions.csv`, `results_row.md` (thời gian train, peak VRAM).

In [ ]:
# === Cell bootstrap (giống nhau ở mọi notebook, xem CODEBASE.md mục 6) ===
import os, subprocess, sys
from pathlib import Path

REPO = "github.com/yunaLee21/Multimodal_Sarcasm_Detection.git"
REF = "thanhtruc/fusion-concat"  # đổi về "main" sau khi PR Phase 4 được merge; hoặc 1 commit hash để tái lặp
CODE_DIR = "/tmp/IE403"    # clone ra ngoài /kaggle/working để token không bị lưu vào output


def get_github_token():
    """Token cho repo private; repo public thì không cần tạo secret."""
    try:
        if os.environ.get("KAGGLE_KERNEL_RUN_TYPE"):
            from kaggle_secrets import UserSecretsClient
            return UserSecretsClient().get_secret("GITHUB_TOKEN")
        from google.colab import userdata
        return userdata.get("GITHUB_TOKEN")
    except Exception:
        return None


ON_CLOUD = bool(os.environ.get("KAGGLE_KERNEL_RUN_TYPE")) or "COLAB_RELEASE_TAG" in os.environ
if ON_CLOUD:
    if "COLAB_RELEASE_TAG" in os.environ and not os.environ.get("KAGGLE_KERNEL_RUN_TYPE"):  # ảnh Kaggle dựng từ ảnh Colab nên cũng có biến này
        from google.colab import drive
        drive.mount("/content/drive")
    token = get_github_token()
    url = f"https://{token}@{REPO}" if token else f"https://{REPO}"
    subprocess.run(["rm", "-rf", CODE_DIR])
    subprocess.run(["git", "clone", "-q", url, CODE_DIR], check=True)
    subprocess.run(["git", "-C", CODE_DIR, "checkout", "-q", REF], check=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", f"{CODE_DIR}/requirements-kaggle.txt"], check=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", CODE_DIR], check=True)
    if CODE_DIR + "/src" not in sys.path:
        sys.path.insert(0, CODE_DIR + "/src")  # kernel đang chạy chưa thấy package vừa cài -e
    os.chdir(CODE_DIR)
elif Path.cwd().name == "notebooks":
    os.chdir(Path.cwd().parent)  # local: chạy từ gốc repo (đã `pip install -e .` một lần)

print("cwd:", os.getcwd())
commit = subprocess.run(["git", "rev-parse", "--short", "HEAD"], capture_output=True, text=True).stdout.strip()
print("code commit:", commit or "unknown")

In [ ]:
# Chạy test của repo trước khi train: vài chục giây, model trong test là bản tí hon nên không tải weight.
# Test fail thì dừng notebook luôn, không tốn GPU train code lỗi. Test cần Java/VnCoreNLP tự skip nếu thiếu.
RUN_TESTS = True
if RUN_TESTS:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "pytest"], check=True)
    result = subprocess.run([sys.executable, "-m", "pytest", "tests", "-q", "-x", "-p", "no:cacheprovider"],
                            capture_output=True, text=True)
    print(result.stdout[-5000:], result.stderr[-2000:])
    if result.returncode != 0:
        raise RuntimeError("pytest fail, xem output phía trên trước khi train")

In [ ]:
import gc
import json
import logging
import re
import shutil
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import display

from vimmsd.utils.config import load_config

logging.basicConfig(level=logging.INFO, format="%(asctime)s %(levelname)s %(message)s", force=True)
logging.getLogger("httpx").setLevel(logging.WARNING)

# Bỏ bớt config nếu chỉ muốn chạy lại một phần. Chạy lần lượt theo thứ tự này.
RUNS = ["text_only_frozen", "image_only_frozen", "fusion_concat_frozen", "fusion_concat_ftlast"]
# True: config đã có test_metrics.json trong output thì bỏ qua (ví dụ attach output version trước rồi chạy tiếp)
SKIP_DONE = True

cfgs = {name: load_config(f"configs/{name}.yaml") for name in RUNS}
OUT = Path(next(iter(cfgs.values())).paths.output_dir)
LABELS = list(next(iter(cfgs.values())).data.labels)
print("output:", OUT)
print("labels:", LABELS)

## Kiểm tra data

Báo lỗi sớm nếu thiếu dataset, thay vì đợi tới lúc train. `stage_preprocessed_cache` là hàm `train.py` cũng gọi: copy `text_preprocessed.json` từ `/kaggle/input` sang `cache_dir` ghi được.

In [ ]:
from vimmsd.data.dataset import stage_preprocessed_cache

cfg0 = next(iter(cfgs.values()))
data_dir = Path(cfg0.paths.data_dir)
if not (data_dir / cfg0.data.train_json).exists():
    raise FileNotFoundError(f"không thấy {data_dir / cfg0.data.train_json}: Add Input hhhoang/vimmsd-dataset")
for path in stage_preprocessed_cache(cfg0):
    meta = json.loads(path.read_text(encoding="utf-8")).get("meta", {})
    print(f"{path.name}: pipeline_key={meta.get('pipeline_key')}, preprocess_version={meta.get('preprocess_version')}")

## Số parameters train được

Build từng model trên CPU để đếm, rồi giải phóng. Dùng cho bảng thống kê tài nguyên.

In [ ]:
from vimmsd.models.classifier import build_model

param_rows = []
for name, cfg in cfgs.items():
    model = build_model(cfg)
    total = sum(p.numel() for p in model.parameters())
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    head = sum(p.numel() for n, p in model.named_parameters() if p.requires_grad and not n.startswith(("text_encoder.", "image_encoder.")))
    param_rows.append({"config": name, "trainable (M)": trainable / 1e6, "trong đó encoder (M)": (trainable - head) / 1e6,
                       "head (M)": head / 1e6, "tổng (M)": total / 1e6})
    del model
    gc.collect()
params = pd.DataFrame(param_rows).set_index("config")
params.round(2)

## Train + đánh giá

Mỗi config: `train.py` (train, chọn best checkpoint theo val macro-F1, chấm test) rồi `evaluate.py --split test` (ghi `test_predictions.csv` để phân tích lỗi).
Checkpoint `last.pt` lưu mỗi epoch. Session bị ngắt thì attach output version trước làm input, copy thư mục `outputs/` về `/kaggle/working`, rồi chạy lại cell này: config đã xong được bỏ qua.

In [ ]:
for name in RUNS:
    out_dir = OUT / name
    if SKIP_DONE and (out_dir / "test_metrics.json").exists() and (out_dir / "test_predictions.csv").exists():
        print(f"=== bỏ qua {name}: đã có kết quả trong {out_dir}")
        continue
    print(f"=== train {name}")
    !python scripts/train.py --config configs/{name}.yaml --note "Phase 4 {name}"
    if not (out_dir / "test_metrics.json").exists():
        raise RuntimeError(f"{name}: train.py không tạo được test_metrics.json, xem log phía trên")
    !python scripts/evaluate.py --config configs/{name}.yaml --split test

## Bảng so sánh

`val`: best val macro-F1 (checkpoint được chọn). `test`: macro-F1/P/R trên tập test chia từ train. Thời gian và peak VRAM đọc từ `results_row.md`.
Baseline Phase 3 (fine-tune toàn bộ encoder) được thêm vào nếu tìm thấy trong `/kaggle/input`, hoặc điền tay vào `MANUAL_REFERENCE`.

In [ ]:
# Baseline Phase 3 nhập tay khi không attach được output, ví dụ:
# MANUAL_REFERENCE = {"image_only (full FT)": {"macro_f1": 0.3945}}
MANUAL_REFERENCE = {}


def parse_results_row(path):
    """| ngày | `config` | `commit` | env | val | test | phút | ghi chú | -> phút, peak VRAM, commit."""
    if not path.exists():
        return {}
    line = path.read_text(encoding="utf-8").strip().splitlines()[-1]  # chạy lại thì dòng mới nhất ở cuối
    cols = [c.strip() for c in line.strip("|").split("|")]  # ghi chú có thể chứa "|" nên chỉ dùng 7 cột đầu
    vram = re.search(r"peak VRAM ([\d.]+) GB", line)
    return {"commit": cols[2].strip("`"), "train (phút)": float(cols[6]),
            "peak VRAM (GB)": float(vram.group(1)) if vram else None}


def metrics_row(label, out_dir, mode):
    test = json.loads((out_dir / "test_metrics.json").read_text())
    val_path = out_dir / "best_val_metrics.json"
    row = {"model": label, "chế độ": mode,
           "val macro-F1": json.loads(val_path.read_text())["macro_f1"] if val_path.exists() else None,
           "test macro-F1": test["macro_f1"], "test macro-P": test["macro_precision"],
           "test macro-R": test["macro_recall"], "accuracy": test["accuracy"]}
    row.update({f"F1 {c}": m["f1"] for c, m in test["per_class"].items()})
    row.update(parse_results_row(out_dir / "results_row.md"))
    return row


MODES = {"text_only_frozen": "đóng băng", "image_only_frozen": "đóng băng",
         "fusion_concat_frozen": "đóng băng", "fusion_concat_ftlast": "mở 2 block cuối"}
rows = [metrics_row(name, OUT / name, MODES.get(name, "")) for name in RUNS if (OUT / name / "test_metrics.json").exists()]

for ref in ("text_only", "image_only"):
    hits = sorted(Path("/kaggle/input").rglob(f"{ref}/test_metrics.json")) if Path("/kaggle/input").exists() else []
    if hits:
        rows.append(metrics_row(f"{ref} (Phase 3)", hits[0].parent, "fine-tune toàn bộ"))
for label, m in MANUAL_REFERENCE.items():
    rows.append({"model": label, "chế độ": "fine-tune toàn bộ", "test macro-F1": m.get("macro_f1")})

table = pd.DataFrame(rows).set_index("model")
table = table.join(params[["trainable (M)"]], how="left")
pd.set_option("display.max_columns", None)
display(table.round(4))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4.5))
macro = table["test macro-F1"].dropna().sort_values()
axes[0].barh(macro.index, macro.values, color=["tab:blue" if "concat" in i else "tab:gray" for i in macro.index])
axes[0].set_xlabel("test macro-F1")
axes[0].set_title("Macro-F1 trên test")
for y, v in enumerate(macro.values):
    axes[0].text(v, y, f" {v:.3f}", va="center")

per_class = table[[f"F1 {c}" for c in LABELS if f"F1 {c}" in table]].dropna(how="all")
per_class.columns = [c.removeprefix("F1 ") for c in per_class.columns]
per_class.T.plot.bar(ax=axes[1], rot=0)
axes[1].set_ylabel("F1")
axes[1].set_title("F1 từng lớp")
axes[1].legend(fontsize=8)
plt.tight_layout()
plt.show()

## Ma trận nhầm lẫn (test)

In [ ]:
done = [name for name in RUNS if (OUT / name / "confusion_matrix_test.png").exists()]
fig, axes = plt.subplots(1, len(done), figsize=(5 * len(done), 4.5), squeeze=False)
for ax, name in zip(axes[0], done):
    ax.imshow(plt.imread(OUT / name / "confusion_matrix_test.png"))
    ax.set_title(name)
    ax.axis("off")
plt.tight_layout()
plt.show()

## Ảnh giúp được gì? So sánh từng mẫu: concat với text-only

Cùng chế độ đóng băng, nên khác biệt chỉ đến từ việc thêm vector ảnh. Đếm mẫu concat đúng mà text-only sai (ảnh giúp) và ngược lại (ảnh làm nhiễu), theo nhãn thật.

In [ ]:
def load_pred(name):
    return pd.read_csv(OUT / name / "test_predictions.csv").set_index("id")


text_pred = load_pred("text_only_frozen")
compare = {}
for fusion_name in ("fusion_concat_frozen", "fusion_concat_ftlast"):
    if not (OUT / fusion_name / "test_predictions.csv").exists():
        continue
    fused = load_pred(fusion_name)
    both = text_pred[["label", "pred", "caption"]].join(fused[["pred"]], rsuffix="_concat")
    both["text đúng"] = both["pred"] == both["label"]
    both["concat đúng"] = both["pred_concat"] == both["label"]
    both["ảnh giúp (concat đúng, text sai)"] = both["concat đúng"] & ~both["text đúng"]
    both["ảnh nhiễu (text đúng, concat sai)"] = both["text đúng"] & ~both["concat đúng"]
    compare[fusion_name] = both
    summary = both.groupby("label")[["text đúng", "concat đúng", "ảnh giúp (concat đúng, text sai)",
                                     "ảnh nhiễu (text đúng, concat sai)"]].sum()
    summary.insert(0, "n", both.groupby("label").size())
    print(f"=== text_only_frozen vs {fusion_name}")
    display(summary)

In [ ]:
# Vài mẫu ảnh giúp: concat đúng, text-only sai
both = compare.get("fusion_concat_frozen")
if both is not None:
    helped = both[both["concat đúng"] & ~both["text đúng"]]
    display(helped.sample(min(10, len(helped)), random_state=0)[["label", "pred", "pred_concat", "caption"]])

## Dòng kết quả cho `reports/results.md`

Copy các dòng dưới vào `reports/results.md` rồi commit. Bảng thống kê tài nguyên lấy cột `train (phút)`, `peak VRAM (GB)` và `trainable (M)` ở bảng so sánh.

In [ ]:
for name in RUNS:
    path = OUT / name / "results_row.md"
    if path.exists():
        print(path.read_text(encoding="utf-8").strip())

## Đọc kết quả

- `fusion_concat_frozen` so với `max(text_only_frozen, image_only_frozen)`: concat có tốt hơn modality mạnh nhất khi chỉ train MLP không. Đây là câu hỏi chính của Phase 4.
- `fusion_concat_ftlast` so với `fusion_concat_frozen`: mở 2 block cuối được thêm bao nhiêu, đổi lại thời gian và VRAM.
- So với Phase 3 (fine-tune toàn bộ): concat đóng băng thấp hơn PhoBERT fine-tune toàn bộ là bình thường; chỉ kết luận về fusion khi so cùng chế độ.
- Lớp `text-sarcasm` có rất ít mẫu nên F1 của lớp này dao động mạnh giữa các lần chạy; nếu còn thời gian, chạy thêm seed (`--override seed=1`) trước khi kết luận.